# get-children-callable-param — ex1: get_children yields (name, value) for Tensor-valued attributes

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `get-children-callable-param`. Running the final beacon cell reports progress against the `Backprop: get_children callable param` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: get_children callable param` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`get-children-callable-param`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "get-children-callable-param"
DD_SUBTOPIC = "Backprop: get_children callable param"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## get_children — nn.Module-style param iteration — quick refresher

`nn.Module` walks its **submodules** + **parameters** so the optimizer can find every trainable tensor. The minimal pattern: scan `__dict__` for any attribute that is a `Tensor` (or another `Module`) and yield `(name, value)`:

```python
def get_children(self):
    for name, val in self.__dict__.items():
        if isinstance(val, (Tensor, Module)):
            yield name, val
```

Three rules:
- **`yield`, don't return a list.** Callers usually want to iterate   once; generators are zero-allocation and compose with `for` cleanly.
- **Skip non-Tensor / non-Module attributes.** A `Linear` layer also   stores `in_features: int`, `out_features: int` — those are config,   not trainable state, and must not show up in the optimizer's param   list.
- **Recurse via submodules' `get_children` (NOT here).** This helper   is the LOCAL step; `parameters()` is the recursive walk built on top   of it. Stay shallow here.

Convention: `name` is the attribute name (e.g. `"weight"`), used later for state-dict keys and debug output.

### Exercise 1 — get_children yields (name, value) for Tensor-valued attributes

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the nn.Module-style get_children pattern: scan __dict__ for MiniTensor-valued attributes and yield (name, tensor) pairs, skipping configuration (ints, strings, layer-shape ints, etc.).
> Keywords: get_children, nn.Module, parameters, yield, isinstance
> ```

**KCs targeted:** `get-children-callable-param`, `parameter-subclass-of-tensor`

We've given you a tiny `Module` base class. Implement `get_children(self)` as a **generator** that yields `(name, value)` for every attribute on `self` whose value `isinstance(_, MiniTensor)`:

```
class Linear(Module):
    def __init__(self):
        self.weight = MiniTensor(t.randn(4, 3), requires_grad=True)
        self.bias   = MiniTensor(t.zeros(4),    requires_grad=True)
        self.in_features = 3   # config: NOT a child
        self.out_features = 4  # config: NOT a child

list(Linear().get_children())
  → [('weight', <MiniTensor ...>), ('bias', <MiniTensor ...>)]
```

Three rules:

**1. `yield`, do not `return` a list.** Generators are zero-allocation and compose with `for` cleanly. The caller usually wants `for name, child in m.get_children(): ...`.

**2. Skip non-MiniTensor attributes.** A `Linear` layer also stores `in_features: int`, `out_features: int` — those are config, not trainable state, and must NOT show up. Use `isinstance(val, MiniTensor)`.

**3. Stay shallow (don't recurse).** This helper is the LOCAL step; recursive walks (`parameters()`) are built on top of it.

Scan `self.__dict__.items()` — that's where Python keeps instance attributes, in insertion order.

In [ ]:
class Module:
    """Tiny nn.Module stand-in. Subclasses set MiniTensor-valued attrs."""
    def get_children(self):
        """Yield (name, value) for every MiniTensor attribute on self."""
        raise NotImplementedError()


def _test_ex1():
    import inspect

    # --- empty module: yields nothing ---
    m_empty = Module()
    assert list(m_empty.get_children()) == [], (
        'empty module must yield nothing'
    )

    # --- it must be a generator (or at least an iterable, not a list) ---
    class L1(Module):
        def __init__(self):
            self.w = MiniTensor(t.randn(2, 3), requires_grad=True)
            self.b = MiniTensor(t.zeros(2),    requires_grad=True)
            self.in_features = 3
            self.out_features = 2

    m = L1()
    children = list(m.get_children())
    names = [n for n, _ in children]
    values = [v for _, v in children]
    assert names == ['w', 'b'], (
        f'should yield only MiniTensor attrs in insertion order, got {names}'
    )
    assert values[0] is m.w, 'value must BE the same object (identity, not copy)'
    assert values[1] is m.b

    # --- config attrs MUST be skipped ---
    names_set = set(names)
    assert 'in_features' not in names_set, (
        'int config attrs must not appear in get_children output'
    )
    assert 'out_features' not in names_set

    # --- mixed-type attrs: only MiniTensor instances are yielded ---
    class Mixed(Module):
        def __init__(self):
            self.name = 'mlp'           # str → skip
            self.dropout_p = 0.1        # float → skip
            self.shape = (3, 4)         # tuple → skip
            self.W = MiniTensor(t.randn(4, 3))   # MiniTensor → KEEP
            self.scratch = t.zeros(4)   # raw torch.Tensor → skip
            self.b = MiniTensor(t.zeros(4))      # MiniTensor → KEEP

    mx = Mixed()
    names = [n for n, _ in mx.get_children()]
    assert names == ['W', 'b'], (
        f'must yield only MiniTensor attrs, in insertion order, got {names}'
    )

    # --- raw torch.Tensor must be SKIPPED (only MiniTensor counts as a child) ---
    assert 'scratch' not in set(names), (
        'raw torch.Tensor must be skipped (only MiniTensor counts as a child)'
    )

    # --- Parameter (Tensor subclass) is also picked up, since isinstance(p, MiniTensor) holds ---
    class Param(MiniTensor):
        def __init__(self, array):
            super().__init__(array, requires_grad=True)

    class WithParam(Module):
        def __init__(self):
            self.p = Param(t.randn(3))
            self.q = MiniTensor(t.randn(3))

    wp = WithParam()
    names = [n for n, _ in wp.get_children()]
    assert names == ['p', 'q'], (
        f'Parameter subclass of MiniTensor must be included, got {names}'
    )

    # --- get_children itself behaves like a generator: returns an iterator object ---
    iterator = m.get_children()
    assert iter(iterator) is iterator or inspect.isgenerator(iterator), (
        'get_children should be a generator / iterator, not a list — '
        'zero-allocation for callers that just want `for ... in`'
        f' (got {type(iterator).__name__})'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class Module:
    def get_children(self):
        for name, val in self.__dict__.items():
            if isinstance(val, MiniTensor):
                yield name, val
```

**Why `__dict__.items()` (not `dir(self)`).** `dir` includes inherited class attributes, methods, dunder fields — all junk for this use case. `__dict__` is exactly the instance attributes the user set in `__init__`, in insertion order (Python 3.7+).

**Why a generator.** A caller that just wants to iterate (`for name, child in m.get_children(): ...`) pays zero allocation cost — no intermediate list. If a caller wants a list, they can always `list(m.get_children())` explicitly.

**Parameter subclassing matters here.** `isinstance(p, MiniTensor)` is True for any `MiniTensor` subclass (including `Parameter`). If `Parameter` were composition-not-inheritance, `get_children` would silently skip every trainable param — which is exactly the reason for the IS-A relationship in the part-3 `parameter-subclass-of-tensor` atom.

**Recursion lives elsewhere.** A full `parameters(recurse=True)` would walk children of children. Keep this primitive shallow — single responsibility, easy to test, easy to compose.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()